# Folding proteins on a MacBook

**Biomolecular structure prediction with ESMFold2 — 25 August 2026**

The same four folds as the GPU notebook — a protein, a protein bound to RNA and
DNA, a drug on its receptor, and a complex with alignments — but on **Apple
Silicon**, through a pure-[MLX](https://github.com/ml-explore/mlx) port running
on the Metal GPU.

No CUDA. No cloud. No network. A laptop on a plane can fold a protein.

**The code is the same as the GPU notebook.** Not similar — the same. The MLX
model is a drop-in for the CUDA one, so every `builder.fold(...)` call below is
the line you would write on an H100.

### What you need

| | |
|---|---|
| **Machine** | Apple Silicon (M1–M4). Intel Macs cannot run this |
| **Memory** | 32 GB minimum, **48 GB comfortable** |
| **Download** | ~25 GB of weights, once |


## Setup

Three installs in a clean environment:

```bash
uv venv --python 3.12 .venv

# 1. the featurizer, decoder, and viewer. Brings torch, rdkit, biotite,
#    py3Dmol and ipywidgets. The CUDA-only extras (xformers, flash-attn,
#    TransformerEngine, cuequivariance) are gated to Linux and are skipped here.
uv pip install --python .venv/bin/python \
  "esm"

# 2. the pure-MLX ESMFold2 + ESMC. --no-deps on purpose: its packaging wants
#    transformers>=5.7 (hence huggingface-hub>=1.0), which collides with the
#    esm featurizer's fork. The MLX modules only need mlx, safetensors,
#    huggingface_hub and numpy at runtime.
uv pip install --python .venv/bin/python --no-deps \
  "mlx-lm @ git+https://github.com/faustomilletari/mlx-lm.git@main"

# 3. something to run this notebook in
uv pip install --python .venv/bin/python jupyterlab

.venv/bin/jupyter lab
```

That is everything. `matplotlib` is optional — only the PAE plot in the GPU
notebook needs it.

Hugging Face auth must be configured; `biohub/ESMFold2`, `biohub/ESMFold2-Fast`
and `biohub/ESMC-6B` may be gated.


In [ ]:
!uv pip install -q "esm"
!uv pip install -q "mlx-lm @ git+https://github.com/faustomilletari/mlx-lm.git@main"

In [ ]:
import mlx.core as mx

print("MLX", mx.__version__)
print(
    f"working set: {mx.metal.device_info()['max_recommended_working_set_size'] / 2**30:.0f} GiB"
)

The display helpers, same file as the GPU notebook.


In [ ]:
import numpy as np
import py3Dmol
from IPython.display import HTML, display

# Same pLDDT bands and colours as cookbook/tutorials/esmfold2.ipynb.
PLDDT_LEGEND = (
    '<span style="color:#FF7D45;">&#9632;</span> &lt;50 &nbsp;'
    '<span style="color:#FFDB13;">&#9632;</span> 50–70 &nbsp;'
    '<span style="color:#65CBF3;">&#9632;</span> 70–90 &nbsp;'
    '<span style="color:#0053D6;">&#9632;</span> &gt;90'
)
CHAIN_COLORS = ["#4A90E2", "#F5A623", "#50E3C2", "#B886D8", "#7FB3E8", "#E8836D"]


def plddt_hex(v):
    """Convert pLDDT score to hex color."""
    if v >= 90:
        return "#0053D6"
    if v >= 70:
        return "#65CBF3"
    if v >= 50:
        return "#FFDB13"
    return "#FF7D45"


def _view(result, width, height):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(result.complex.to_mmcif(), "mmcif")
    return view


def show_plddt(result, width=700, height=480):
    """Cartoon coloured by per-residue confidence."""
    view = _view(result, width, height)
    view.setStyle({}, {})
    for i, score in enumerate(np.asarray(result.plddt).ravel() * 100):
        view.setStyle({"resi": i + 1}, {"cartoon": {"color": plddt_hex(score)}})
    view.addStyle({"hetflag": True}, {"stick": {}})
    view.zoomTo()
    display(HTML(PLDDT_LEGEND))
    return view


def show_chains(result, width=700, height=480):
    """One colour per chain; ligands and ions drawn as sticks."""
    view = _view(result, width, height)
    for i, chain in enumerate(sorted(result.complex.metadata.chain_lookup.values())):
        color = CHAIN_COLORS[i % len(CHAIN_COLORS)]
        view.setStyle({"chain": chain}, {"cartoon": {"color": color}})
    view.addStyle({"hetflag": True}, {"stick": {}})
    view.zoomTo()
    return view

## Load the model

Two imports: the **model** from `mlx_lm`, the **builder** from `esm`. That is the
only line in this notebook that differs from the CUDA version.

The first load pulls ~25 GB and takes a few minutes; after that it is resident.


In [ ]:
from mlx_lm.models.esmfold2 import ESMFold2Model  # <- MLX instead of esm

from esm.models.esmfold2 import ESMFold2InputBuilder

model = ESMFold2Model.from_pretrained("biohub/ESMFold2-Fast")
builder = ESMFold2InputBuilder()

We use `num_loops=3, num_sampling_steps=50` rather than the GPU notebook's
`10 / 100`. Both work; these keep each fold to seconds on a laptop instead of
minutes. Nothing else changes.


## 1. The pattern

Three steps, exactly as on a GPU:

1. **Say what you want** — a `StructurePredictionInput` listing your molecules.
2. **Fold it** — `builder.fold(model, ...)`.
3. **Look at it.**


In [ ]:
import time

from esm.models.esmfold2 import ProteinInput, StructurePredictionInput

UBIQUITIN = (
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
)

spi = StructurePredictionInput(sequences=[ProteinInput(id="A", sequence=UBIQUITIN)])

t0 = time.perf_counter()
result = builder.fold(model, spi, num_loops=10, num_sampling_steps=100)
print(f"{len(UBIQUITIN)} residues in {time.perf_counter() - t0:.1f} s")
print(f"pLDDT {float(result.plddt.mean()):.3f}   pTM {float(result.ptm):.3f}")

show_plddt(result)

A protein folded on a laptop, offline, in a couple of seconds — and the same
confidence numbers you would get on a datacentre GPU.

**Blue is confident, red-orange is not.** The tail at the end runs orange, and it
should: that tail really is floppy in solution.

Everything below changes only **step 1** — what goes in the list.


## 2. Add more molecules

RNase H1 (PDB 4H8K) grabs an RNA/DNA hybrid and cuts the RNA strand. It works as
a pair of identical copies — that is what `id=["A", "B"]` means: **one sequence,
present twice**.


In [ ]:
from esm.models.esmfold2 import DNAInput, RNAInput

RNASEH = (
    "MNKIIIYTDGGARGNPGPAGIGVVITDEKGNTLHESSAYIGETTNNVAEYEALIRALEDLQ"
    "MFGDKLVDMEVEVRMNSELIVRQMQGVYKVKEPTLKEKFAKIAHIKMERVPNLVFVHIPRE"
    "KNARADELVNEAIDKALS"
)

spi = StructurePredictionInput(
    sequences=[
        ProteinInput(id=["A", "B"], sequence=RNASEH),  # the enzyme, two copies
        RNAInput(id="C", sequence="CGACACCUGAUUCC"),  # the strand it cuts
        DNAInput(id="D", sequence="GGAATCAGGTGTCG"),  # its partner strand
    ]
)

t0 = time.perf_counter()
result = builder.fold(model, spi, num_loops=3, num_sampling_steps=50)
print(f"folded in {time.perf_counter() - t0:.1f} s")
print(f"pLDDT {float(result.plddt.mean()):.3f}   ipTM {float(result.iptm):.3f}")

show_chains(result)

Two protein chains with the RNA/DNA duplex threaded through the middle.

A third number appeared: **ipTM**. Whenever there is more than one molecule, this
is the one to read — it says whether they are positioned correctly against each
other, not just folded correctly on their own.


## 3. Add a drug

Semaglutide bound to its receptor. Three new things, one line each:

- an **unnatural amino acid** at position 1, named by its PDB code `AIB`
- a **fatty-acid tail** that is not a protein at all, drawn as SMILES
- that tail is **chemically bonded** to the peptide, so we say which atoms join

To keep this quick on a laptop we use the receptor's **extracellular domain**
(116 residues) rather than the full 490-residue protein. It is the part that
grips the peptide.


In [ ]:
from esm.models.esmfold2 import CovalentBond, LigandInput, Modification

RECEPTOR = (
    "MKTIIALSYIFCLVFADYKDDDDLEVLFQGPARPQGATVSLWETVQKWREYRRQCQRSLTEDPPPATDLFCNRTFDEYAC"
    "WPDGEPGSFVNVSCPWYLPWASSVPQGHVYRFCTAEGLWLQKDNSSLPWRDLSECEESKRGERSSPEEQLLFLYIIYTVG"
    "YALSFSALVIASAILLGFRHLHCTRNYIHLNLFASFILRALSVFIKDAALKWMYSTAAQQHQWDGLLSYQDSLSCRLVFL"
    "LMQYCVAANYYWLLVEGVYLYTLLAFSVFSEQWIFRLYVSIGWGVPLLFVVPWGIVKYLYEDEGCWTRNSNMNYWLIIRL"
    "PILFAIGVNFLIFVRVICIVVSKLKANLMCKTDIKCRLAKSTLTLIPLLGTHEVIFAFVMDEHARGTLRFIKLFTELSFT"
    "SFQGLMVAILYCFVNNEVQLEFRKSWERWRLEHLHIQRDSSMKPLKCPTSSLSSGATAGSSMYTATCQASCSPAGLEVLF"
    "QGPHHHHHHH"
)
PEPTIDE = "HAEGTFTSDVSSYLEGQAAKEFIAWLVRGRG"
FATTY_TAIL = (
    "C(=O)(CCOCCOCC(=O)NCCOCCOCCNCC(=O)N[C@@H](CCC(=O)NCCCCCCCCCCCCCCCCCC(=O)O)C(=O)O)"
)

lysine = PEPTIDE.index("K")  # the tail attaches here

spi = StructurePredictionInput(
    sequences=[
        ProteinInput(id="A", sequence=RECEPTOR),
        ProteinInput(
            id="B",
            sequence=PEPTIDE,
            modifications=[Modification(position=1, ccd="AIB")],
        ),
        LigandInput(id="C", smiles=FATTY_TAIL),
    ],
    covalent_bonds=[
        CovalentBond(
            chain_id1="B",
            res_idx1=lysine,
            atom_idx1=8,  # the lysine's N
            chain_id2="C",
            res_idx2=0,
            atom_idx2=0,
        )  # the tail's first atom
    ],
)

t0 = time.perf_counter()
result = builder.fold(model, spi, num_loops=3, num_sampling_steps=50)
print(f"folded in {time.perf_counter() - t0:.1f} s")
print(f"pLDDT {float(result.plddt.mean()):.3f}   ipTM {float(result.iptm):.3f}")

show_chains(result)

The peptide sits against the receptor domain with the fatty tail trailing off it.
That tail is why semaglutide is injected weekly rather than daily — it sticks to
a blood protein and slows the drug's clearance.


## 4. Add evolution

You can also hand the model an **alignment** — the same protein from hundreds of
other species. If two positions always mutate together across evolution they are
probably touching, and for two chains that also hints at how they meet.

**One thing to know:** the `-Fast` checkpoint has no alignment reader and will
ignore an MSA without complaining. For alignments, load the full model.


In [ ]:
model = ESMFold2Model.from_pretrained("biohub/ESMFold2")

In [ ]:
BASE = "https://raw.githubusercontent.com/Biohub/esm/main/cookbook/tutorials"
!curl -sL {BASE}/g3l5_chainA.a3m -o chainA.a3m
!curl -sL {BASE}/g3l5_chainB.a3m -o chainB.a3m

In [ ]:
from esm.utils.msa import MSA

# A two-protein complex from vaccinia virus (PDB 7YTU), one alignment per chain.
CHAIN_A = (
    "GPYYPTNKLQAAVMETDRENAIIRQRNDEIPTRTLDTAIFTDASTVASAQIHLYYNSNIGKII"
    "MSLNGKKHTFNLYDDNDIRTLLPILLLSK"
)
CHAIN_B = (
    "GPNMFFMPKRKIPDPIDRLRRANLACEDDKLMIYGLPWMTTQTSALSINSKPIVYKDCAKLLRSINGSQPVSLNDVLRR"
)

spi = StructurePredictionInput(
    sequences=[
        ProteinInput(id="A", sequence=CHAIN_A, msa=MSA.from_a3m(path="chainA.a3m")),
        ProteinInput(id="B", sequence=CHAIN_B, msa=MSA.from_a3m(path="chainB.a3m")),
    ]
)

t0 = time.perf_counter()
result = builder.fold(model, spi, num_loops=10, num_sampling_steps=100)
print(f"folded in {time.perf_counter() - t0:.1f} s")
print(f"pLDDT {float(result.plddt.mean()):.3f}   ipTM {float(result.iptm):.3f}")

show_chains(result)

An alignment is just a text file. Loading it is one line; attaching it is one argument.


## 5. What you just did

Four structures on a laptop, offline, nothing sent anywhere — using the same code
you would run on a datacentre GPU:

```python
spi = StructurePredictionInput(sequences=[ ...your molecules... ])
result = builder.fold(model, spi)
```

### What it costs

Mean fold time in seconds, ESMFold2-Fast at 50 sampling steps:

| Length | M4 Pro (MLX) | L40 reference | L40 fused kernels |
|---:|---:|---:|---:|
| 100 | 2.4 | 0.6 | 0.6 |
| 300 | 15.8 | 2.6 | 1.0 |
| 500 | 42.9 | 9.0 | 2.2 |
| 1000 | 189.3 | 121.9 | 9.1 |

The fair comparison is against *reference* CUDA, since that is the same
arithmetic: at 1000 residues a laptop is within about 1.5x of an L40. The large
gap is only against fused Triton kernels, which MLX has no equivalent of.

**Up to roughly 1000 residues, on battery, offline, a Mac is a real folding
machine.** Past that, use CUDA or the API.

Peak memory stays near 14 GiB for small inputs and reaches ~41 GiB at 1000
residues, which is why 48 GB is the comfortable size.


## 6. If you want to go further

- **A desktop app.** [iESMFold2](https://github.com/faustomilletari/iESMFold2)
  wraps this in a native macOS app — sequence boxes, an interactive viewer, a
  model picker.
- **Ask for several answers at once.** `num_diffusion_samples=N` returns N
  structures; rank them by confidence and keep the best.
- **Think harder.** `num_loops` controls how many refinement passes the model
  makes.
- **Longer chains.** Watch memory; the full model near 1000 residues will use
  most of a 48 GB machine.

**Links** — [MLX port](https://github.com/faustomilletari/mlx-lm) ·
[desktop app](https://github.com/faustomilletari/iESMFold2) ·
[esm](https://github.com/Biohub/esm) ·
[MLX](https://github.com/ml-explore/mlx)
